# Name That LLM: Transformer Feature Extraction (Colab T4 GPU)

This notebook extracts high-dimensional multi-layer hidden state representations from **DeBERTa-v3** for the Kaggle **Name That LLM** competition.

### Features:
1. **T4 GPU Optimized:** Runs in `torch.float16` with length-sorted batching for maximum throughput.
2. **Multi-Layer Representation:** Mean-pools the last 4 hidden layers excluding padding tokens.
3. **NaN & Stability Guard:** Automatically recovers with fp32 if non-finite logits/representations occur.
4. **Direct Local Download:** Automatically downloads `train_transformer_embeddings.npy` and `test_transformer_embeddings.npy` to drop straight into your local `artifacts/` folder.

### Step 1: Verify Hardware Accelerator (NVIDIA T4 GPU)

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
if not torch.cuda.is_available():
    print("\n[WARNING] GPU is NOT enabled! Please go to Runtime -> Change runtime type -> Select T4 GPU.")
else:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[SUCCESS] Active GPU: {device_name} ({vram_gb:.2f} GB VRAM)")

### Step 2: Install Required Libraries

In [ ]:
!pip install -q transformers accelerate datasets

### Step 3: Load Data (`train.csv` and `test.csv`)
Upload `train.csv` and `test.csv` if they are not already in your working directory.

In [ ]:
import os
import pandas as pd
from pathlib import Path

if not os.path.exists("train.csv"):
    print("Please upload train.csv:")
    from google.colab import files
    uploaded = files.upload()

if not os.path.exists("test.csv"):
    print("Please upload test.csv (or a sample test.csv):")
    from google.colab import files
    uploaded = files.upload()

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")
print(f"Loaded train.csv with {len(df_train)} rows across {df_train['prompt_id'].nunique()} unique prompts.")
print(f"Loaded test.csv with {len(df_test)} rows.")

### Step 4: Multi-Layer Transformer Extractor Implementation
Uses multi-layer frozen hidden states, length-sorted batching, and fp16.

In [ ]:
import gc
import time
from typing import List, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

class MultiLayerTransformerExtractor:
    """
    Extracts multi-layer hidden states from a transformer backbone.
    - Runs in torch.float16 on CUDA.
    - Uses length-sorted batching to maximize speed and minimize padding.
    - Features automatic fp32 fallback if non-finite representations appear.
    """
    def __init__(
        self,
        model_name: str = "microsoft/deberta-v3-base",
        num_layers: int = 4,
        max_length: int = 512,
        batch_size: int = 16,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        use_fp16: bool = True,
    ):
        self.model_name = model_name
        self.num_layers = num_layers
        self.max_length = max_length
        self.batch_size = batch_size
        self.device = torch.device(device)
        self.use_fp16 = use_fp16 and (self.device.type == "cuda")
        self.model = None
        self.tokenizer = None
        self.hidden_dim = None
        self.output_dim = None

    def load_model(self):
        print(f"Loading {self.model_name}...")
        config = AutoConfig.from_pretrained(self.model_name, output_hidden_states=True)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=True)
        
        dtype = torch.float16 if self.use_fp16 else torch.float32
        self.model = AutoModel.from_pretrained(self.model_name, config=config, torch_dtype=dtype)
        self.model.eval()
        self.model.to(self.device)
        for p in self.model.parameters():
            p.requires_grad = False
        
        self.hidden_dim = config.hidden_size
        self.output_dim = self.hidden_dim * self.num_layers
        print(f"Model loaded. Hidden dim: {self.hidden_dim}, Extracted dim (x{self.num_layers} layers): {self.output_dim}")

    def _pool_hidden_states(self, hidden_states: Tuple[torch.Tensor, ...], attention_mask: torch.Tensor) -> torch.Tensor:
        layers_to_pool = hidden_states[-self.num_layers:]
        pooled = []
        mask_exp = attention_mask.unsqueeze(-1).expand_as(layers_to_pool[0]).float()
        mask_sum = mask_exp.sum(dim=1).clamp(min=1e-9)
        for layer in layers_to_pool:
            layer_f = layer.float()
            mean_p = torch.sum(layer_f * mask_exp, dim=1) / mask_sum
            pooled.append(mean_p)
        return torch.cat(pooled, dim=-1)

    def extract(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            self.load_model()
        
        n = len(texts)
        if n == 0:
            return np.empty((0, self.output_dim), dtype=np.float32)

        # Length-sorted batching
        lengths = [len(t) for t in texts]
        sorted_idx = np.argsort(lengths)[::-1]
        sorted_texts = [texts[i] for i in sorted_idx]

        all_embs = [None] * n
        t0 = time.time()
        
        for i in range(0, n, self.batch_size):
            batch_texts = sorted_texts[i : i + self.batch_size]
            batch_idx = sorted_idx[i : i + self.batch_size]
            
            encoded = self.tokenizer(batch_texts, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            
            with torch.no_grad():
                try:
                    out = self.model(input_ids=input_ids, attention_mask=attention_mask)
                    pooled = self._pool_hidden_states(out.hidden_states, attention_mask)
                    if not torch.isfinite(pooled).all():
                        # fp32 fallback
                        m32 = self.model.float()
                        out = m32(input_ids=input_ids, attention_mask=attention_mask)
                        pooled = self._pool_hidden_states(out.hidden_states, attention_mask)
                        if self.use_fp16:
                            self.model.half()
                    batch_embs = pooled.cpu().numpy().astype(np.float32)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    gc.collect()
                    # single fallback
                    batch_list = []
                    for single_t in batch_texts:
                        enc_s = self.tokenizer([single_t], padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
                        out_s = self.model(input_ids=enc_s["input_ids"].to(self.device), attention_mask=enc_s["attention_mask"].to(self.device))
                        p_s = self._pool_hidden_states(out_s.hidden_states, enc_s["attention_mask"].to(self.device))
                        batch_list.append(p_s.cpu().numpy().astype(np.float32))
                    batch_embs = np.vstack(batch_list)

            for orig_i, emb in zip(batch_idx, batch_embs):
                all_embs[orig_i] = emb
            
            if (i // self.batch_size) % 20 == 0 or (i + self.batch_size >= n):
                elapsed = time.time() - t0
                done = min(i + self.batch_size, n)
                speed = done / max(0.01, elapsed)
                print(f"Extracted {done}/{n} samples ({speed:.1f} samples/s)...")

        return np.vstack(all_embs)

### Step 5: Extract Representations for Train and Test

In [ ]:
# Initialize extractor (default: microsoft/deberta-v3-base; can also use microsoft/deberta-v3-large)
extractor = MultiLayerTransformerExtractor(
    model_name="microsoft/deberta-v3-base",
    num_layers=4,
    max_length=512,
    batch_size=16,
)

print("\n--- Extracting Train Embeddings ---")
train_texts = df_train["text"].fillna("").tolist()
train_embeddings = extractor.extract(train_texts)
print(f"Train embeddings extracted: {train_embeddings.shape}, dtype: {train_embeddings.dtype}")

print("\n--- Extracting Test Embeddings ---")
test_texts = df_test["text"].fillna("").tolist()
test_embeddings = extractor.extract(test_texts)
print(f"Test embeddings extracted: {test_embeddings.shape}, dtype: {test_embeddings.dtype}")

# Sanity checks
assert not np.isnan(train_embeddings).any(), "NaNs found in train embeddings!"
assert not np.isnan(test_embeddings).any(), "NaNs found in test embeddings!"
print("\n[SUCCESS] All embeddings validated: finite and clean.")

### Step 6: Quick 5-Fold GroupKFold CV Benchmark
Let's immediately verify the authorship attribution strength of these transformer representations using a fast Ridge / Logistic Regression baseline on Colab!

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

target_models = sorted(df_train["model"].unique().tolist())
model_to_idx = {m: i for i, m in enumerate(target_models)}
y = np.array([model_to_idx[m] for m in df_train["model"]])
groups = df_train["prompt_id"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_embeddings)

gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(df_train))

print("Running fast 5-fold GroupKFold CV on DeBERTa embeddings...")
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_scaled, y, groups)):
    clf = LogisticRegression(C=0.5, max_iter=500, random_state=42)
    clf.fit(X_scaled[tr_idx], y[tr_idx])
    oof_preds[val_idx] = clf.predict(X_scaled[val_idx])

score = balanced_accuracy_score(y, oof_preds)
print(f"\n>>> DeBERTa-v3 Hidden State 5-Fold GroupKFold Balanced Accuracy: {score:.4f} <<<")

### Step 7: Save and Download `.npy` Embeddings
These `.npy` files should be placed into your local `artifacts/` folder.

In [ ]:
from google.colab import files

# Save locally in Colab runtime
np.save("train_transformer_embeddings.npy", train_embeddings)
np.save("test_transformer_embeddings.npy", test_embeddings)
print("Saved files:")
print(f" - train_transformer_embeddings.npy: {os.path.getsize('train_transformer_embeddings.npy') / (1024**2):.2f} MB")
print(f" - test_transformer_embeddings.npy: {os.path.getsize('test_transformer_embeddings.npy') / (1024**2):.2f} MB")

print("\nTriggering browser download...")
files.download("train_transformer_embeddings.npy")
files.download("test_transformer_embeddings.npy")
print("Download initiated! Once finished, move these two files to your project's artifacts/ folder.")